# **K-means-based Quantization** vs **Linear Quantization**

**Linear quantization** maps values with a simple formula such as **q = round(x / scale) + zero_point** (uniform step size). It is usually implemented as symmetric or affine/asymmetric quantization, often per-tensor or per-channel. **K-means-based quantization** is non-uniform: it clusters weights or activations into a learned codebook of centroids, then stores centroid indices instead of the original values. In model compression literature, this is often called codebook quantization, weight sharing, or LUT-based quantization.

## Accuracy

- **Linear Quantization**: usually very competitive at 8-bit, especially with per-channel scaling and/or QAT. On mainstream models and toolchains, INT8 linear quantization often preserves most accuracy while remaining easy to deploy. TensorFlow’s guidance and examples show that full-integer INT8 can keep accuracy close to FP32, though some architectures are sensitive and may need QAT.

- **K-means Quantization**: often has an advantage at **very low bit-widths** because the learned centroids can fit non-Gaussian or heavy-tailed weight distributions better than uniform bins. That is why classic compression work and newer low-bit papers keep revisiting codebook-based quantization.

## Latency / Throughput

- **Linear Quantization**: usually wins in real deployment because it maps directly onto efficient INT8/INT4 GEMM and convolution kernels. PyTorch and TFLite both describe this as enabling integer compute paths and faster inference.

- **K-means Quantization**: often loses on standard runtimes because inference needs centroid lookup and frequently some form of dequantization or nonstandard execution. Recent papers explicitly frame this as the core problem and propose custom lookup-table kernels or custom accelerators to recover speed, which tells you the baseline hardware path is not naturally optimized for it.

## Hardware support

- **Linear Quantization**: strongest by far. CPUs, NPUs, DSPs, Edge TPU–style integer accelerators, TFLite, ONNX Runtime, and PyTorch quantized backends are all built around uniform integer arithmetic with scale/zero-point metadata. This is the default production path.

- **K-means Quantization**: weak native support. You usually need custom kernels, LUT-aware runtimes, or specialized hardware ideas to get the theoretical benefit. Without that, the format is more of a **compression format** than a universally efficient **execution format**.

## Model size / memory

- **Linear Quantization**: simple and compact; metadata overhead is tiny. INT8 gives the expected ~4x shrink from FP32 when applied broadly.

- **K-means Quantization**: can be even better for **weight storage**, especially when many values can share a small codebook. That is the appeal of weight sharing in Deep Compression. But you must also store codebooks and indices, and runtime memory behavior may be worse if repeated LUT accesses become the bottleneck.

## Training / calibration complexity

- **Linear Quantization**: easier. PTQ and QAT are standard, well-supported workflows. Calibration datasets and debugging tools are mature.

- **K-means Quantization**: typically more complex. You need clustering/codebook construction, centroid assignment, sometimes retraining or centroid fine-tuning, and more custom handling at export/inference time.

## Where each one is strongest

- **Choose linear quantization** when you care about production inference speed, broad hardware compatibility, easy export, and stable tooling. This is usually the right default for mobile, edge, server inference, and standard ML deployment stacks.

- **Choose k-means quantization** when memory is extremely constrained, bit-width is very low, weight distributions are badly matched by uniform bins, or you are willing to use custom kernels / specialized runtimes to chase extra compression or low-bit accuracy.

Rule of thumb is:

- **For real products**: start with linear quantization.
- **For research or extreme compression**: investigate k-means/codebook quantization.
- **For low-bit deployment without custom kernels**: linear is usually safer.
- **For lowest-bit accuracy under tight memory budgets**: k-means may be better, but only if you can afford the runtime complexity